# ⚙️ Production Model Training in One Notebook

This notebook teaches the production mindset without splitting the work into separate Python scripts.

We will cover:
- why notebooks need structure in production
- clean function design inside one notebook
- config-driven training using a dictionary
- logging and model artifact saving
- a full training workflow in one place

## Why notebooks need structure

Notebook code is powerful for exploration, but hidden state and hard-coded values can make it fragile in production.

A strong notebook workflow still needs:
- clear sections
- reusable functions
- explicit configuration
- reproducible evaluation
- logged progress

In [ ]:
import matplotlib.pyplot as plt

# Simple conceptual comparison: notebook vs production-style notebook workflow
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].text(
    0.5, 0.5,
    'Notebook\n\nExploration\nHidden state\nHard-coded params\nManual runs',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#E94B4B', alpha=0.2)
)
axes[0].set_title('Loose Notebook Workflow')
axes[0].axis('off')

axes[1].text(
    0.5, 0.5,
    'Structured Notebook\n\nConfig\nReusable functions\nLogging\nModel + metrics artifacts',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='#4A90D9', alpha=0.2)
)
axes[1].set_title('Production-Style Notebook Workflow')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Single responsibility inside a notebook

The same design principle that applies to scripts can also apply to notebook cells and helper functions.

We will keep each function focused on a single job:
- create the data
- split into training/test
- train the model
- evaluate the model
- save the model and metrics

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
import json
import logging
from pathlib import Path

# Configuration: this replaces a separate YAML config file for this notebook example
config = {
    'target_column': 'churn',
    'test_size': 0.2,
    'random_state': 42,
    'model_type': 'logistic_regression',
    'max_iter': 500,
    'artifact_dir': 'artifacts'
}

# 1) Load / create data

def create_synthetic_data():
    X, y = make_classification(
        n_samples=1000,
        n_features=6,
        n_informative=4,
        n_redundant=1,
        n_classes=2,
        weights=[0.65, 0.35],
        random_state=config['random_state']
    )
    feature_names = ['age', 'monthly_spend', 'tenure_months', 'support_tickets', 'usage_score', 'contract_type']
    df = pd.DataFrame(X, columns=feature_names)
    df['contract_type'] = df['contract_type'].round(0).astype(int)
    df['churn'] = y
    return df

# 2) Train/test split

def split_data(df, target_column, test_size, random_state):
    X = df.drop(columns=[target_column])
    y = df[target_column]
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)

# 3) Train the model

def train_model(X_train, y_train):
    model = LogisticRegression(max_iter=config['max_iter'], random_state=config['random_state'])
    model.fit(X_train, y_train)
    return model

# 4) Evaluate the model

def evaluate_model(model, X_test, y_test):
    preds = model.predict(X_test)
    metrics = {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'report': classification_report(y_test, preds, output_dict=True)
    }
    return metrics

# 5) Save artifact

def save_artifacts(model, metrics):
    artifact_dir = Path(config['artifact_dir'])
    artifact_dir.mkdir(exist_ok=True, parents=True)

    joblib.dump(model, artifact_dir / 'production_model.joblib')
    (artifact_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2, default=str), encoding='utf-8')

    print(f'Saved model to {artifact_dir / "production_model.joblib"}')
    print(f'Saved metrics to {artifact_dir / "metrics.json"}')

# Build a run workflow

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('artifacts/training.log', mode='a', encoding='utf-8'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info('Starting notebook-based training workflow')

df = create_synthetic_data()
X_train, X_test, y_train, y_test = split_data(
    df,
    config['target_column'],
    config['test_size'],
    config['random_state']
)

model = train_model(X_train, y_train)
metrics = evaluate_model(model, X_test, y_test)
save_artifacts(model, metrics)

logger.info('Training complete')
print(metrics)

## Logging and artifact saving

A production workflow does not rely on ad hoc output alone. We log the run and save the trained model plus evaluation metrics to disk.

This is the same principle as a script-based pipeline, just implemented inside one notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simple loss curve: demonstrates training progress and overfitting awareness
epochs = np.arange(1, 11)
train_loss = [0.72, 0.56, 0.42, 0.35, 0.30, 0.25, 0.23, 0.21, 0.20, 0.19]
val_loss = [0.75, 0.51, 0.39, 0.32, 0.30, 0.31, 0.36, 0.42, 0.47, 0.52]

plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss, label='Train loss', color='#4A90D9', linewidth=2)
plt.plot(epochs, val_loss, label='Validation loss', color='#E94B4B', linewidth=2)
plt.axvline(5.5, color='gray', linestyle='--', linewidth=1.5, label='Best epoch')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Why this is still production-minded

Even inside a notebook, you can follow the same core discipline as a deployable training workflow:
- keep parameters in one place
- write reusable functions
- log results
- save artifacts
- keep evaluation explicit

This is the bridge between exploratory coding and real ML engineering.

# ✅ Key Takeaways

- A notebook can still follow production-style discipline without separate script files.
- Reusable functions make a notebook easier to maintain and interpret.
- Config dictionaries help make training runs reproducible.
- Logging and artifact saving are essential for production-minded ML.
- A well-structured notebook is a practical step toward real-world ML systems.

## Quiz

1. Why do notebooks still need structure even when they are not deployed as scripts?
2. What is the purpose of keeping configuration in one dictionary?
3. Why is logging more useful than raw print statements in training?
4. What should a notebook save after a training run?
5. How do functions help you stay organized in a notebook workflow?